In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle
import cudf

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

/opt/conda/envs/rapids-25.02/lib/python3.10/site-packages/cudf/pandas/__init__.py:65: UserWarning: cudf.pandas detected an already configured memory resource, ignoring 'CUDF_PANDAS_RMM_MODE'=managed_pool
  warnings.warn(


In [ ]:
%LoadCheckpoint /home/sahilbhatia/pandax/dias_notebooks/nickwan/creating-player-stats-using-tracking-data/src/small_bench/checkpoints/post_cell_9.pickle

In [4]:
%%RecordEvent
%%time
### cell 10 ###

# cell 10 – optimized for cuDF
# compute max LOS diff for offense and min LOS diff for defense, then average per player
off = (
    df[df['posTeam'] == 1]
      [['gameId','playId','nflId','los_diff']]
      .groupby(['gameId','playId','nflId'], as_index=False)
      .max()
)
def1 = (
    df[df['posTeam'] == 0]
      [['gameId','playId','nflId','los_diff']]
      .groupby(['gameId','playId','nflId'], as_index=False)
      .min()
)
# use GPU‐side concat instead of pandas append
los_diff = pd.concat([off, def1], ignore_index=True)
# average LOS diff per nflId
top = (
    los_diff[['nflId','los_diff']]
      .groupby('nflId', as_index=False)
      .mean()
)
# merge back onto snap_speed and rename columns
snap_speed = (
    snap_speed
      .merge(top)
      .rename(columns={
          's':'snap_s',
          'a':'snap_a',
          'los_diff':'snap_los_diff'
      })
)

CPU times: user 52.3 ms, sys: 8.04 ms, total: 60.3 ms
Wall time: 60.3 ms


In [ ]:
%Checkpoint /home/sahilbhatia/pandax/dias_notebooks/nickwan/creating-player-stats-using-tracking-data/src/rewritten/o4_mini_high_small/checkpoints/post_cell_10_try_0.pickle